# Analyse FAIRness


In [34]:
from earthcode.fairtool import add_fairtool_results_to_product
import pystac
from pathlib import Path
import os
import json
from earthcode.fairtool import product_audit_to_fair_dict, analyse_product
import pystac
pystac.set_stac_version('1.0.0')

In [ ]:
# product_dir = pystac.Catalog.from_file('https://esa-earthcode.github.io/open-science-catalog-metadata/products/catalog.json')
# product_dir

<Catalog id=products>

(do not run if you already have the scores as it will rescore everything)

In [37]:
# # Path('./open-science-catalog-metadata/products/')

# for target_product in product_dir.get_children():

#     # read local file
#     file_dir = Path('./open-science-catalog-metadata/products/') / f'{target_product.id}/collection.json'
#     with open(file_dir, 'r', encoding='utf-8') as f:
#         product_collection = json.load(f)
#         product = pystac.Collection.from_dict(product_collection,
#                                                 migrate=False,
#                                                 root=None,
#                                                 preserve_dict=True)
#     product.set_self_href(str(file_dir.resolve()))

#     # do fair analysis
#     result = analyse_product(target_product, timeout=15, seed=123)

#     # if no cloud assets, skip
#     # if result.cloud_score == 0.0: 
#     #     continue

#     # add to score to local catalog
#     print(file_dir)
#     result_dict = product_audit_to_fair_dict(result)
#     product = product_collection.copy()
#     for k,v in result_dict.items():
#         product[k] = v

#     with open(file_dir, 'w', encoding='utf-8') as f:
#         json.dump(product, f, ensure_ascii=False, indent=2)


### 1. Collect the FAIR data

In [44]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from osc_fairness_stats import FAIR_METRICS, PRINCIPLE_COLORS

catalog = Path("open-science-catalog-metadata/products")
data = pd.DataFrame([json.loads(p.read_text()) for p in sorted(catalog.glob("*/collection.json"))])
fair = data[[metric.key for metric in FAIR_METRICS]]
total = len(data)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "figure.dpi": 120, "figure.constrained_layout.use": True,
    "font.size": 11, "axes.titlesize": 16, "axes.titleweight": "bold",
    "axes.titlelocation": "left", "axes.titlepad": 18,
    "axes.edgecolor": "white", "axes.grid": False,
    "text.color": "#1e293b", "ytick.color": "#475569",
})
print(f"Collected {total} products and {fair.shape[1]} FAIR metrics.")

Collected 355 products and 20 FAIR metrics.


In [ ]:
data[(data['fair:Reusable_cloud_assets_rate'] == 1.0)].id.values

### 2. FAIR metrics — number / total

In [ ]:
counts = fair.gt(0).sum()
counts = counts[counts < total]
metrics = [metric for metric in FAIR_METRICS if metric.key in counts.index]
labels = [f"{metric.principle}: {metric.label}" for metric in metrics]
colours = [PRINCIPLE_COLORS[metric.principle] for metric in metrics]

fig, ax = plt.subplots(figsize=(12, 6))
ax.barh(labels, [total] * len(counts), color="#f1f5f9", height=0.6)
ax.barh(labels, counts, color=colours, height=0.6)
for row, count in enumerate(counts):
    ax.text(total * 1.03, row, f"{count} / {total}", va="center")
ax.invert_yaxis()
ax.set(title="FAIR metrics · below full coverage", xlim=(0, total * 1.21), xticks=[])
plt.show()

### 3. Number of products with a visualisation


In [ ]:
counts = fair["fair:Reusable_has_visualisation"].value_counts().reindex([True, False], fill_value=0)
counts.index = ["With visualisation", "Without visualisation"]

ax = counts.plot.barh(figsize=(9, 2.8), color=["#0f766e", "#cbd5e1"], width=0.55)
ax.bar_label(ax.containers[0], labels=[f"{n} / {total}" for n in counts], padding=8)
ax.invert_yaxis()
ax.set(title="Products with a visualisation", xlim=(0, total * 1.21), xticks=[], ylabel="")
plt.show()

### 4. Licenses — proprietary vs not proprietary


In [ ]:
licenses = data["license"].str.strip().fillna("Missing")
proprietary = licenses.eq("proprietary")
counts = pd.Series({"Proprietary": proprietary.sum(), "Not proprietary": (~proprietary).sum()})

print(f"FAIR license flag: {fair['fair:Reusable_has_license'].eq(True).sum()} / {total} marked True")
ax = counts.plot.barh(figsize=(9, 2.8), color=["#b45309", "#2563eb"], width=0.55)
ax.bar_label(ax.containers[0], labels=[f"{n} / {total}" for n in counts], padding=8)
ax.invert_yaxis()
ax.set(title="Actual license values · proprietary vs not proprietary", xlim=(0, total * 1.21), xticks=[], ylabel="")
plt.show()

display(licenses.value_counts().rename_axis("License").to_frame("Products"))
display(data.loc[proprietary, ["id", "license"]].reset_index(drop=True))

### 5. All metrics

In [ ]:
from matplotlib.patches import Patch

scores = fair[[m.key for m in FAIR_METRICS]].mean()
fig, ax = plt.subplots(figsize=(13, 7))
wedges, _ = ax.pie([1] * len(scores), startangle=90, counterclock=False,
    colors=[PRINCIPLE_COLORS[m.principle] for m in FAIR_METRICS],
    wedgeprops={"width": 0.45, "edgecolor": "white"})
for wedge, score in zip(wedges, scores):
    wedge.set_alpha(0.15 + 0.85 * score)
ax.text(0, 0.06, f"{scores.mean():.1%}", ha="center", va="center", fontsize=44, weight="bold")
ax.text(0, -0.17, "overall mean", ha="center", fontsize=16)

legend = []
for principle, colour in PRINCIPLE_COLORS.items():
    score = scores[[m.key for m in FAIR_METRICS if m.principle == principle]].mean()
    legend.append(Patch(color=colour, label=f"{principle}  {score:.1%}"))
ax.legend(handles=legend, loc="center left", bbox_to_anchor=(1, 0.5), frameon=False,
    fontsize=24, handlelength=2.2, handleheight=1.2, labelspacing=1.0)
ax.set_title("Overall FAIR assessment", fontsize=24)
plt.show()